---
# PARTIE 2 -- L'API DataFrame

## 2.1 Du RDD au DataFrame

L'API RDD est puissante mais verbeuse et peu optimisee : Spark ne sait pas ce que
contiennent les elements (ils sont des `object` Python opaques). L'API **DataFrame**
introduit un schema explicite, ce qui permet a l'optimiseur Catalyst de generer
un plan d'execution bien plus efficace.

### Creation depuis un RDD


In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, TimestampType
)

# Schema explicite -- toujours preferable a l'inference automatique
schema_velib = StructType([
    StructField("station_id",       IntegerType(), nullable=False),
    StructField("nom_station",      StringType(),  nullable=True),
    StructField("code_arr",         IntegerType(), nullable=True),
    StructField("capacite",         IntegerType(), nullable=True),
    StructField("velos_meca",       IntegerType(), nullable=True),
    StructField("velos_elec",       IntegerType(), nullable=True),
    StructField("bornettes_libres", IntegerType(), nullable=True),
    StructField("horodatage",       StringType(),  nullable=True),
])

# Convertir le RDD de dicts en RDD de Row, puis en DataFrame
row_rdd = step3.map(lambda d: Row(**d))
df_from_rdd = spark.createDataFrame(row_rdd, schema=schema_velib)

df_from_rdd.printSchema()
df_from_rdd.show(5, truncate=False)
print(f"Lignes : {df_from_rdd.count():,}  --  Partitions : {df_from_rdd.rdd.getNumPartitions()}")


---
## 2.2 Lecture directe depuis les fichiers Parquet

En pratique, on ne passe presque jamais par les RDD pour creer un DataFrame.
Spark peut lire directement de nombreux formats (CSV, JSON, Parquet, Delta, ORC...).

Le format **Parquet** est le format de facto pour les donnees analytiques :
- Stockage colonnaire (lecture partielle possible)
- Compression integree (Snappy, Zstandard...)
- Schema embarque dans les fichiers
- Partitionnement natif (Hive-style)


In [ ]:
# Lecture des fichiers Parquet pre-prepares (partitionnes par annee et mois)
# Spark detecte automatiquement le schema et les colonnes de partition
df_velib = #

df_velib.printSchema()
print(f"\nDimensions : {df_velib.count():,} lignes x {len(df_velib.columns)} colonnes")
print(f"Partitions : {df_velib.rdd.getNumPartitions()}")


In [ ]:
# Lecture selective d'une partition (predicate pushdown)
# Spark ne lira que les fichiers du dossier annee=2023/mois=06
df_juin_2023 = #
print(f"Juin 2023 : {df_juin_2023.count():,} lignes")

# Comparer avec une lecture complete puis filtre -- le plan d'execution est identique
# grace au predicate pushdown, mais on le verifie avec explain()
print("\n--- Plan d'execution (filtre apres lecture) ---")
df_velib.filter("annee = 2023 AND mois = 6").explain(mode="formatted")


---
## 2.3 Exploration et diagnostic des donnees

Avant tout traitement, il faut comprendre la structure et la qualite des donnees.
Spark propose des fonctions d'agregation statistiques integrees.


In [ ]:
from pyspark.sql import functions as F

# Vue d'ensemble statistique
print("=== Statistiques descriptives (colonnes numeriques) ===")
# TODO

In [ ]:
# Comptage des valeurs nulles par colonne
print("=== Valeurs nulles par colonne ===")
#

In [ ]:
# Detection des anomalies evidentes
print("=== Anomalies detectees ===")

# Stations avec capacite <= 0
# TODO
print(f"  Snapshots avec capacite <= 0          : {n_cap_zero:,}")

# Stations avec plus de velos que de capacite
# TODO
print(f"  Snapshots avec total > capacite + 2   : {n_debord:,}")

# Valeurs negatives
for col_name in ["velos_meca", "velos_elec", "bornettes_libres"]:
    # TODO
    print(f"  {col_name:<30} < 0 : {n_neg:,}")


In [ ]:
# Distribution temporelle : combien de snapshots par mois ?
print("=== Couverture temporelle ===")
(
# TODO
)


---
## 2.4 Nettoyage et construction des features

Nous allons maintenant construire la table `disponibilite` propre, telle que
definie dans le schema cible du projet.


In [ ]:
from pyspark.sql.functions import (
    to_timestamp, col, when, round as spark_round,
    year, month, dayofweek, hour, expr
)

# ── 1. Parsing et typage du timestamp ───────────────────────────────────────
df_clean = # TODO

# ── 2. Suppression des lignes avec timestamp invalide ───────────────────────
# TODO

# ── 3. Suppression des stations avec capacite invalide ──────────────────────
# TODO

# ── 4. Correction des valeurs negatives (bornage a 0) ───────────────────────
# TODO

# ── 5. Calcul du taux d'occupation ──────────────────────────────────────────
# TODO

# ── 6. Bornage du taux entre 0 et 1 ─────────────────────────────────────────
# TODO

# ── 7. Extraction de features temporelles (utiles pour le ML en Jour 3) ─────
# Ajouter les colonnes : année, moois, jour_sem, heure, is_weekend

print(f"Lignes apres nettoyage : {df_clean.count():,}")
df_clean.printSchema()
df_clean.show(3, truncate=True)


In [ ]:
# [EXERCICE]
# Ajoutez une colonne "statut" de type StringType avec les valeurs :
#   "vide"   si taux_occupation < 0.1
#   "plein"  si taux_occupation > 0.9
#   "normal" sinon
#
# Indice : utilisez F.when(...).when(...).otherwise(...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
df_clean = df_clean.withColumn(
    "statut",
    # ...
    F.lit("TODO")   # remplacer cette ligne
)

df_clean.groupBy("statut").count().orderBy("statut").show()


---
## 2.5 Chargement des donnees meteorologiques

La source meteorologique est le fichier SYNOP de Meteo-France pour la station
Paris-Montsouris, telechargeable librement sur data.gouv.fr.
Le format SYNOP est semi-structure et necessite quelques transformations.

Voici le format des colonnes pertinentes :

| Colonne SYNOP | Signification | Unite |
|---------------|---------------|-------|
| `AAAAMMJJHH`  | Horodatage    | UTC   |
| `T`           | Temperature   | K (Kelvin) |
| `U`           | Humidite relative | % |
| `FF`          | Vitesse du vent | m/s |
| `RR1`         | Precipitation | mm/h |
| `N`           | Nebulosity (couverture nuageuse) | octals (0-8) |


In [ ]:
# Lecture du CSV avec separateur point-virgule
# Le fichier contient des en-tetes sur plusieurs lignes -- on saute les deux premieres
schema_meteo = StructType([
    StructField("NUM_POSTE",   StringType(),  True),
    StructField("NOM_USUEL",   StringType(),  True),
    StructField("LAT",         DoubleType(),  True),
    StructField("LON",         DoubleType(),  True),
    StructField("ALTI",        DoubleType(),  True),
    StructField("AAAAMMJJHH",  StringType(),  True),
    StructField("T",           DoubleType(),  True),
    StructField("U",           DoubleType(),  True),
    StructField("FF",          DoubleType(),  True),
    StructField("RR1",         DoubleType(),  True),
    StructField("N",           DoubleType(),  True),
])

df_meteo_raw = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .option("nullValue", "mq")    # "mq" = manquant dans SYNOP
    .schema(schema_meteo)
    .csv(str(METEO_CSV))
)

print(f"Lignes meteo brutes : {df_meteo_raw.count():,}")
df_meteo_raw.show(5)


In [ ]:
from pyspark.sql.functions import (
    to_timestamp, substring, concat_ws,
    lit, lpad
)

# Transformation du champ AAAAMMJJHH en timestamp
# Format : "2022010100" = 2022-01-01 a 00h UTC
# Sélectionner "horodatage_meteo", "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"
# Supprimer les lignes vides
df_meteo = # ?

# Ajout de l'indicateur pluie
df_meteo = df_meteo.withColumn(
# TODO
)

print(f"Lignes meteo nettoyees : {df_meteo.count():,}")
df_meteo.show(8)
df_meteo.describe(["temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"]).show()


---
## 2.6 Jointure temporelle : Velib' x Meteo

Les observations Velib' sont a la minute, les donnees SYNOP a l'heure.
Pour joindre les deux tables, on **tronque** les horodatages Velib' a l'heure la plus
proche, puis on effectue une jointure sur cette cle commune.


In [ ]:
from pyspark.sql.functions import date_trunc, broadcast

# ── 1. Cle de jointure : heure tronquee ──────────────────────────────────────
df_velib_join = df_clean.withColumn(
    "heure_tronquee",
    # TODO

df_meteo_join = df_meteo.withColumnRenamed(
    "horodatage_meteo", "heure_tronquee"
)

# ── 2. Broadcast join ─────────────────────────────────────────────────────────
# La table meteo est petite (~17 500 lignes pour 2 ans d'observations horaires).
# Avec broadcast(), Spark envoie une copie complete a chaque executeur,
# ce qui evite entierement le shuffle de la grosse table Velib'.
# Regles empiriques : utiliser broadcast() si la petite table < quelques centaines de MB.

df_joint = # ?

# Verification
n_avant  = df_clean.count()
n_apres  = df_joint.count()
n_meteo_null = df_joint.filter(col("temperature_c").isNull()).count()

print(f"Lignes avant jointure  : {n_avant:,}")
print(f"Lignes apres jointure  : {n_apres:,}")
print(f"Snapshots sans meteo   : {n_meteo_null:,} ({100*n_meteo_null/n_apres:.1f}%)")


In [ ]:
# Examinons le plan d'execution pour verifier que le broadcast est bien utilise
print("=== Plan d'execution de la jointure ===")
df_joint.explain(mode="formatted")
# Cherchez "BroadcastHashJoin" dans le plan -- pas de "SortMergeJoin" (qui implique un shuffle)


In [ ]:
# [EXERCICE]
# Calculez, pour chaque combinaison (arrondissement, est_pluie),
# le taux_occupation moyen et le nombre de snapshots.
# Triez par arrondissement croissant, puis par est_pluie.
#
# Indice : utilisez groupBy(...).agg(F.mean(...).alias(...), F.count("*")...)
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :


---
## 2.7 Persistance en memoire : `.cache()` et `.persist()`

Nous allons utiliser `df_joint` intensivement pendant le reste du cours.
Mettons-le en cache pour eviter de recalculer la jointure a chaque action.


In [ ]:
# .cache() == .persist(StorageLevel.MEMORY_AND_DISK)
# Si les donnees ne tiennent pas en RAM, Spark deborde sur disque.

t0 = time.perf_counter()
df_joint.cache()
df_joint.count()   # force la materialisation du cache
t_cache = time.perf_counter() - t0
print(f"Mise en cache (premiere passe) : {t_cache:.2f} s")

# Deuxieme passe -- depuis le cache
t0 = time.perf_counter()
df_joint.count()
t_hit = time.perf_counter() - t0
print(f"Lecture depuis le cache        : {t_hit:.2f} s")
print(f"Gain                           : x{t_cache/t_hit:.1f}")
print()
print("Allez dans Spark UI -> Storage pour voir la taille du cache.")


In [ ]:
# StorageLevel disponibles (par ordre de rapidite / consommation memoire)
from pyspark import StorageLevel

print("StorageLevels disponibles :")
print("  MEMORY_ONLY          : RAM uniquement (eviction si plein)")
print("  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]")
print("  DISK_ONLY            : Disque uniquement (lent mais stable)")
print("  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)")
print("  OFF_HEAP             : Memoire hors JVM (necessite configuration)")
print()
print("Regle pratique :")
print("  - Petits DataFrames reutilises souvent -> MEMORY_ONLY")
print("  - Gros DataFrames reutilises -> MEMORY_AND_DISK")
print("  - Ne pas cacher si utilise une seule fois -> overhead inutile")


---
## 2.8 Ecriture en Parquet partitionne

La table `df_joint` est la table centrale du projet. Nous allons la persister
sur disque en Parquet, partitionne par annee et par mois, pour que les
traitements des jours suivants n'aient pas a la reconstruire.


In [ ]:
OUTPUT_VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"

# Selection finale des colonnes (on elimine les colonnes intermediaires)
colonnes_finales = [
    "station_id", "nom_station", "code_arr", "capacite",
    "horodatage", "velos_meca", "velos_elec", "bornettes_libres",
    "taux_occupation", "statut",
    "annee", "mois", "jour_sem", "heure", "est_weekend",
    "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie"
]

df_sortie = df_joint.select(*colonnes_finales)

t0 = time.perf_counter()
(
    # Ecrire au format Parquet en réécrivant sur les données existantesn et ne parittionnant sur les années et les mois
)
t_write = time.perf_counter() - t0
print(f"Ecriture en {t_write:.1f} s")

# Verification : lecture selective d'une partition
df_verif = spark.read.parquet(str(OUTPUT_VELIB_CONSOLIDE)).filter("annee = 2022 AND mois = 1")
print(f"Janvier 2022 : {df_verif.count():,} lignes")
print(f"Schema : {df_verif.columns}")


In [ ]:
# Comparaison de la taille CSV vs Parquet
import os

def taille_dossier_mb(path: Path) -> float:
    """
    Calcule la taille d'un dossier en octets.
    """
    # TODO
t_csv   = taille_dossier_mb(VELIB_RAW_DIR)
t_parq  = taille_dossier_mb(OUTPUT_VELIB_CONSOLIDE)
ratio   = t_csv / t_parq if t_parq > 0 else 0

print(f"Taille CSV bruts (compresses .gz) : {t_csv:.1f} MB")
print(f"Taille Parquet consolide          : {t_parq:.1f} MB")
print(f"Rapport CSV/Parquet               : x{ratio:.1f}")
print()
print("Le Parquet est plus petit car :")
print("  1. Stockage colonnaire : on compresse des valeurs homogenes")
print("  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)")
print("  3. Run-Length Encoding sur les colonnes de partition (annee, mois)")


---
## Bilan du Jour 1

### Ce que nous avons fait

| Etape | API | Concept cle |
|-------|-----|-------------|
| Chargement CSV brut | RDD | `sc.textFile()`, parsing manuel |
| Filtrage et transformation | RDD | `map()`, `filter()`, evaluation paresseuse |
| Agregation par cle | RDD | `reduceByKey()` vs `groupByKey()` |
| Profil horaire | RDD | `flatMap()`, calcul de moyenne distribuee |
| Jointure RDD | RDD | `join()` sur paires (cle, valeur) |
| Lecture Parquet | DataFrame | `spark.read.parquet()`, predicate pushdown |
| Exploration statistique | DataFrame | `describe()`, comptage de nulls |
| Nettoyage | DataFrame | `dropna()`, `when()`, `greatest()` |
| Features temporelles | DataFrame | `year()`, `hour()`, `dayofweek()` |
| Jointure broadcast | DataFrame | `broadcast()`, eviter le shuffle |
| Persistance | DataFrame/RDD | `.cache()`, `.persist()`, `StorageLevel` |
| Ecriture partitionnee | DataFrame | `write.partitionBy().parquet()` |

### Concepts fondamentaux a retenir

1. **Evaluation paresseuse** : les transformations construisent un plan, les actions l'executent.
2. **Le DAG** : lire le Spark UI est une competence essentielle, pas optionnelle.
3. **reduceByKey > groupByKey** : toujours combiner localement avant de shuffler.
4. **DataFrame > RDD** : l'optimiseur Catalyst rend les DataFrames significativement plus
   rapides que les RDD pour les memes operations.
5. **Cache strategique** : ne mettre en cache que ce qui est reutilise plusieurs fois.
6. **Broadcast join** : quand une table est petite, eviter le shuffle de la grosse table.

### Pour demain (Jour 2)

La table `disponibilite_consolidee.parquet` sera le point de depart du Jour 2.
Nous allons l'interroger avec Spark SQL (fenetrage temporel, requetes analytiques),
puis la connecter a un flux simule de mises a jour en temps reel avec Structured Streaming.


In [ ]:
# Nettoyage de fin de session
# Toujours liberer la SparkSession proprement pour liberer les ressources
print("Arret de la SparkSession...")
spark.stop()
print("SparkSession arretee. Bonne nuit !")
